# Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

**Design overview**

The agent uses the `Agent` class from `lib/agents.py`, which runs an LLM ↔ tool loop and keeps
conversation history per `session_id`. It is given four tools:

| Tool | Purpose |
|---|---|
| `retrieve_game` | Semantic search over the local ChromaDB collection built in Part 1 |
| `evaluate_retrieval` | LLM-as-judge: are the retrieved documents enough to answer? |
| `game_web_search` | Tavily web search, used only when local knowledge is not enough |
| `store_game_fact` | Saves a verified fact found on the web back into ChromaDB, so it is answered locally next time |

The agent's text answer is then converted into a structured `GameAnswer` object.

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import hashlib
from typing import Literal

import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
# Load API keys from the .env file
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")

# Fail early with a clear message instead of a confusing error later on
missing = [name for name, value in {
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "TAVILY_API_KEY": TAVILY_API_KEY,
    "CHROMA_OPENAI_API_KEY": CHROMA_OPENAI_API_KEY,
}.items() if not value]
if missing:
    raise EnvironmentError(f"Missing keys in .env: {', '.join(missing)}")

**Configuration choices**

- **`gpt-4o-mini`**: supports tool calling and structured outputs, and is cheap and fast. It is also the default in `lib/llm.py`.
- **Temperature 0**: this is a factual research agent, so we want consistent answers rather than creative ones.
  The `Agent` class defaults to 0.7, so we override it.
- **Same ChromaDB path, collection name and embedding model as Part 1**: queries must be embedded with the
  same model as the stored documents, otherwise similarity scores are meaningless.
- **`get_collection` instead of `create_collection`**: Part 2 only reads the existing database; it must never wipe it.
- **Part 2 never modifies the `udaplay` collection.** Facts learned from the web go into a separate
  `udaplay_web` collection. An earlier version saved them into `udaplay` and deleted them in a reset
  step; after that delete, searches started returning the wrong games (Wii Sports for a Pokémon
  question), sending every question to the web. Keeping the trusted Part 1 data read-only avoids this.
- **Health check:** searching for a game we know is in the DB must return it first. If not, the
  notebook stops with instructions instead of silently answering everything from the web.

In [4]:
MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0.0

# Open the database created in Part 1 (same path, name and embedding model)
chroma_client = chromadb.PersistentClient(
    path="chromadb",
    settings=Settings(anonymized_telemetry=False),
)

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=CHROMA_OPENAI_API_KEY,
    model_name="text-embedding-ada-002",
)

collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

# Facts learned from the web are kept in a separate collection (see store_game_fact)
web_collection = chroma_client.get_or_create_collection(
    name="udaplay_web",
    embedding_function=embedding_fn,
)

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

print(f"Loaded 'udaplay' with {collection.count()} documents "
      f"and 'udaplay_web' with {web_collection.count()} learned facts")

# Health check: a game we know is in the DB must come back first.
# If it doesn't, the search index is out of sync and every question would go to the web.
check = collection.query(query_texts=["Pokémon Gold and Silver"], n_results=1)
top_hit = check["metadatas"][0][0]["Name"] if check["metadatas"][0] else None
if top_hit != "Pokémon Gold and Silver":
    raise RuntimeError(
        f"Vector search looks broken (top hit: {top_hit}). "
        "Shut down this kernel, re-run the Part 1 notebook to rebuild the database, "
        "then restart this notebook."
    )
print("Health check passed: vector search returns the expected game.")

Loaded 'udaplay' with 15 documents and 'udaplay_web' with 1 learned facts
Health check passed: vector search returns the expected game.


#### Reset learned facts

Facts learned from the web persist between runs. Dropping the `udaplay_web` collection here, before any
tool is tested, means every run of this notebook starts from the same knowledge (the 15 original games),
so results are reproducible. The original `udaplay` collection is not touched.

This is done in Setup rather than just before the agent tests: in an earlier run, the quick evaluator
test below ran *before* the reset and found a Mortal Kombat X fact left over from the previous run.

In [5]:
chroma_client.delete_collection(name="udaplay_web")
web_collection = chroma_client.create_collection(name="udaplay_web", embedding_function=embedding_fn)

print(f"Original games: {collection.count()}")        # expected: 15
print(f"Learned facts:  {web_collection.count()}")    # expected: 0

Original games: 15
Learned facts:  0


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web

Plus one extra tool, `store_game_fact`, to store useful information for future use.

**Error handling:** every tool catches its own exceptions and returns an `{"error": ...}` result.
`Agent` runs tools directly, so an uncaught exception (network failure, bad API key) would crash the whole
run. Returning the error instead lets the agent tell the user what went wrong or try another tool.

#### Retrieve Game Tool

- Returns the full game record (from the metadata stored in Part 1) plus the **distance** score, so the
  evaluator can see how close each match really was.
- Returns the top 3 matches: enough to cover comparison questions ("which was first?") without flooding
  the model with irrelevant games.
- Searches **both** the original games (`udaplay`) and facts learned from the web (`udaplay_web`), then
  keeps the 3 closest overall. Both use the same embedding model, so their distances are comparable.
- Each result carries a ready-made **`citation`**, e.g. `Local DB: Pokémon Gold and Silver (Game Boy Color, 1999)`,
  or for facts learned from the web, the fact plus its source URL. Answers from the local database can
  then cite the exact records they used, not only web answers.

In [6]:
@tool
def retrieve_game(query: str) -> list[dict]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    - distance: how far the result is from the query (lower = more relevant)
    - citation: how to cite this record in an answer
    """
    try:
        games = []
        for col in (collection, web_collection):
            n = min(3, col.count())  # querying an empty collection can raise an error
            if n == 0:
                continue
            results = col.query(query_texts=[query], n_results=n)
            for meta, distance in zip(results["metadatas"][0], results["distances"][0]):
                game = dict(meta)
                game["distance"] = round(distance, 3)
                label = f"{game.get('Name')} ({game.get('Platform')}, {game.get('YearOfRelease')})"
                if game.get("Source") == "web":
                    game["citation"] = f"Learned from web: {label} - {game.get('SourceURL')}"
                else:
                    game["citation"] = f"Local DB: {label}"
                games.append(game)
    except Exception as e:
        return [{"error": f"Vector DB search failed: {e}"}]

    # Keep the 3 closest matches across both collections
    return sorted(games, key=lambda g: g["distance"])[:3]

In [7]:
# Quick test: should return Pokémon Gold and Silver first
retrieve_game("When was Pokémon Gold and Silver released?")

[{'Platform': 'Game Boy Color',
  'Genre': 'Role-playing',
  'YearOfRelease': 1999,
  'Name': 'Pokémon Gold and Silver',
  'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.',
  'Publisher': 'Nintendo',
  'distance': 0.226,
  'citation': 'Local DB: Pokémon Gold and Silver (Game Boy Color, 1999)'},
 {'Publisher': 'Nintendo',
  'Platform': 'Game Boy Advance',
  'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.',
  'YearOfRelease': 2002,
  'Name': 'Pokémon Ruby and Sapphire',
  'Genre': 'Role-playing',
  'distance': 0.294,
  'citation': 'Local DB: Pokémon Ruby and Sapphire (Game Boy Advance, 2002)'},
 {'Platform': 'Nintendo 64',
  'Genre': 'Platformer',
  'Name': 'Super Mario 64',
  'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.",
  'Publisher': 'Nintendo',
  'YearOfRelease': 1996,
  '

#### Evaluate Retrieval Tool

- Uses a separate LLM call as a **judge**. Deciding whether documents answer a question is a judgement
  call that a similarity score alone can't make (e.g. Mortal Kombat X isn't in the DB, but the closest
  matches still have a distance score).
- `EvaluationReport` is defined here because it is not provided in `lib/`. Passing it as
  `response_format` makes the model return valid JSON with exactly these fields, so we never have to
  parse free text.
- The judge prompt asks for strictness: if the documents only *partly* answer the question, it should
  say so. A false "useful" means a wrong answer; a false "not useful" only costs one web search.
- The judge is told **not to demand more detail than the question asks for**. An earlier, stricter
  version rejected "1999" for "When was Pokémon Gold and Silver released?" because it wasn't an exact
  date, which sent a question the local DB could answer to the web.

In [8]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are enough to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")


EVALUATOR_PROMPT = (
    "You are an expert evaluator for a video game research assistant. "
    "Your task is to evaluate if the documents are enough to respond the query. "
    "Mark them as useful if they contain the facts needed to answer the question as it was asked. "
    "Do not require more detail than the question asks for: for 'when was X released' questions, "
    "the year of release is enough unless the user explicitly asks for an exact date or a specific region. "
    "Mark them as not useful if the game, platform or fact asked about is not in the documents. "
    "Give a detailed explanation, so it's possible to take an action to accept it or not."
)


@tool
def evaluate_retrieval(question: str, retrieved_docs: list[str]) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    try:
        judge = LLM(model=MODEL_NAME, temperature=0.0)
        docs_text = "\n".join(f"- {doc}" for doc in retrieved_docs)
        response = judge.invoke(
            [
                SystemMessage(content=EVALUATOR_PROMPT),
                UserMessage(content=f"Question: {question}\n\nDocuments:\n{docs_text}"),
            ],
            response_format=EvaluationReport,
        )
        report = EvaluationReport.model_validate_json(response.content)
        return report.model_dump()
    except Exception as e:
        return {"useful": False, "description": f"Evaluation failed: {e}"}

In [9]:
# Quick test: the first should be useful, the second should not
docs = [str(g) for g in retrieve_game("Pokémon Gold and Silver release")]
print(evaluate_retrieval("When was Pokémon Gold and Silver released?", docs))

docs = [str(g) for g in retrieve_game("Mortal Kombat X PlayStation 5")]
print(evaluate_retrieval("Was Mortal Kombat X released for PlayStation 5?", docs))

{'useful': True, 'description': 'The documents contain the necessary information to answer the question about the release date of Pokémon Gold and Silver. Specifically, one of the documents explicitly states that Pokémon Gold and Silver was released in 1999. This is sufficient to respond to the query, as it only asks for the year of release.'}
{'useful': False, 'description': "The documents provided do not contain any information regarding the release of Mortal Kombat X for PlayStation 5. They only mention other games such as Marvel's Spider-Man and Halo Infinite, which are unrelated to Mortal Kombat X. Therefore, the documents are not useful for answering the question."}


#### Game Web Search Tool

- `search_depth="advanced"` gives more relevant results for specific factual questions
  (release dates, platforms) at a small extra cost.
- `include_answer=True` asks Tavily for a short summary answer, which is often all the agent needs.
- Each result's content is trimmed to 500 characters and limited to 5 results. Tool output is added to
  the conversation history and resent on every LLM call, so keeping it short saves tokens and cost.
- URLs are kept so the final answer can cite its sources.
- The latest results are also kept in `LAST_WEB_RESULTS`, so `store_game_fact` can check a fact
  against the actual sources before saving it.

In [10]:
# Latest web search results, used by store_game_fact to fact-check before saving
LAST_WEB_RESULTS = {}


@tool
def game_web_search(question: str) -> dict:
    """
    Web search: searches the internet for up-to-date information about video games.
    Use it only when the local vector DB does not have enough information.
    args:
    - question: a question about game industry.

    Returns a short answer summary and a list of results, each with title, url and content.
    """
    try:
        response = tavily_client.search(
            query=question,
            search_depth="advanced",
            include_answer=True,
            max_results=5,
        )
    except Exception as e:
        return {"error": f"Web search failed: {e}"}

    global LAST_WEB_RESULTS
    LAST_WEB_RESULTS = {
        "answer": response.get("answer"),
        "results": [
            {
                "title": r.get("title"),
                "url": r.get("url"),
                "content": (r.get("content") or "")[:500],
            }
            for r in response.get("results", [])
        ],
    }
    return LAST_WEB_RESULTS

In [11]:
# Quick test
game_web_search("Was Mortal Kombat X released for PlayStation 5?")

{'answer': "Mortal Kombat X was originally released for PlayStation 4 on April 14, 2015. While it is playable on PS5, the game's inclusion in the PlayStation Plus Collection for PS5 suggests it will be available to next-generation console players who subscribe to PS Plus. The game's Kombat Pack is also available for both PS5 and PS4, indicating its continued support and availability on these platforms.",
 'results': [{'title': 'PS5 PlayStation Plus Collection To Include Mortal Kombat X - Mortal Kombat Online',
   'url': 'https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V',
   'content': "Mortal Kombat X will be one of the future-retro perks of subscribing to PlayStation Plus on the next gen console. The 2015 MK sequel was one of the highest selling installments of the franchise, and continues to be a favourite among modern kombat enthusiasts. It's included in the PlayStation Plus Collection as one of PS4's defining fighting 

#### Store Game Fact Tool (store useful information for future use)

- Saves facts found on the web into the **`udaplay_web` collection**, which `retrieve_game` also
  searches. It is kept separate from the Part 1 data so the trusted data is never modified, and so the
  learned facts can be cleared without touching it. The collection is persistent, so the next time (even after a restart) the question is answered locally
  without a web search. The library's `LongTermMemory` class was not used because it keeps data in memory
  only and wipes it on start-up.
- It is a **tool the agent chooses to call**, rather than automatically saving every web result. Raw
  search snippets are noisy; letting the agent save one clean, structured record per game keeps the
  database the same shape as the Part 1 data.
- Records use the **same fields and text format as Part 1**, plus `Source` and `SourceURL`, so stored
  facts are searchable in exactly the same way and it is always clear where a fact came from.
- The document ID is built from the game name and platform, and `upsert` is used, so saving the same
  game twice updates it instead of creating a duplicate.
- **Guard in code:** it refuses to store a game + platform that already exists in the original Part 1
  data. An earlier run saved a second Pokémon Gold and Silver record with the North American date
  (2000), which conflicted with the original (1999). The original data is the trusted source, so the
  check is done in Python rather than left to the model.
- The `platform` argument must be the platform the game was **released on**, not one it is merely
  playable on. An earlier run stored Mortal Kombat X as a PlayStation 5 game because it is playable
  there via the PlayStation Plus Collection, and a later question was then answered incorrectly.
- **Fact check before saving.** Despite the instructions, a later run still saved Mortal Kombat X as a
  PlayStation 5 game from 2020. Instructions alone don't reliably stop this, so the tool now checks the
  record against the latest web results (`LAST_WEB_RESULTS`) with a separate LLM call before saving.
  If the sources don't support the name, platform and year, it refuses and explains why, so the agent
  can retry once with corrected values. This costs one extra LLM call, but only when something is saved.

In [12]:
class FactCheck(BaseModel):
    supported: bool = Field(description="Whether the sources support the record as written")
    reason: str = Field(description="Why, including the correct platform/year if the sources show them")


FACT_CHECK_PROMPT = (
    "You check facts before they are saved to a video game database. "
    "Compare the record with the web search results. The record is supported only if the sources show "
    "that the game was released on that platform in that year. A platform where the game is only "
    "playable through backward compatibility, streaming or a subscription collection (such as the "
    "PlayStation Plus Collection) does NOT count as a release on that platform. "
    "If the record is not supported, explain why and give the correct platform and year if the sources show them."
)


def fact_check(name: str, platform: str, year_of_release: int) -> FactCheck:
    """Check a game record against the latest web search results."""
    if not LAST_WEB_RESULTS:
        return FactCheck(supported=False, reason="No web search results to check against. Search first.")
    checker = LLM(model=MODEL_NAME, temperature=0.0)
    response = checker.invoke(
        [
            SystemMessage(content=FACT_CHECK_PROMPT),
            UserMessage(content=(
                f"Record: {name} was released on {platform} in {year_of_release}.\n\n"
                f"Web search results: {json.dumps(LAST_WEB_RESULTS, ensure_ascii=False)}"
            )),
        ],
        response_format=FactCheck,
    )
    return FactCheck.model_validate_json(response.content)


@tool
def store_game_fact(
    name: str,
    platform: str,
    year_of_release: int,
    description: str,
    genre: str,
    publisher: str,
    source_url: str,
) -> dict:
    """
    Saves a verified fact about a game, found with game_web_search, into the local vector DB
    so it can be answered without a web search next time.
    Only use it for facts that are clearly supported by the web search results.
    args:
    - name: name of the game
    - platform: platform the game was officially released on (e.g. PlayStation 4). Not a platform
      it is only playable on through backward compatibility or a subscription collection.
    - year_of_release: year the game was released on that platform
    - description: short description including the key fact that was found
    - genre: genre of the game (use "Unknown" if not known)
    - publisher: publisher of the game (use "Unknown" if not known)
    - source_url: URL of the web page that supports the fact
    """
    try:
        # Never overwrite or duplicate the original Part 1 data: it is the trusted source
        existing = collection.get(where={"$and": [{"Name": name}, {"Platform": platform}]})
        if existing["ids"]:
            return {"stored": False, "reason": "This game and platform are already in the local database."}

        # Only save facts the web sources actually support
        check = fact_check(name, platform, year_of_release)
        if not check.supported:
            return {"stored": False, "reason": f"Fact check failed: {check.reason}"}

        game = {
            "Name": name,
            "Platform": platform,
            "Genre": genre,
            "Publisher": publisher,
            "Description": description,
            "YearOfRelease": int(year_of_release),
            "Source": "web",
            "SourceURL": source_url,
        }
        # Same text format as Part 1, so stored facts are retrieved the same way
        content = f"[{platform}] {name} ({year_of_release}) - {description}"
        # Stable ID from name + platform: saving the same game twice updates it
        key = f"{name}|{platform}".lower().encode("utf-8")
        doc_id = "web-" + hashlib.md5(key).hexdigest()[:12]

        web_collection.upsert(ids=[doc_id], documents=[content], metadatas=[game])
        return {"stored": True, "id": doc_id, "learned_facts": web_collection.count()}
    except Exception as e:
        return {"stored": False, "error": f"Could not store fact: {e}"}

### Agent

**Instructions:** the system prompt spells out the workflow step by step. The model decides *when* to
call each tool, so clear rules are what keep it consistent: always check local knowledge first, always
evaluate, only search the web when needed, and only store facts that came from the web.

It also asks the agent to say where the answer came from and to cite URLs, which makes answers
checkable and feeds the structured output below.

In [13]:
INSTRUCTIONS = """
You are UdaPlay, an AI research assistant for the video game industry.
You answer questions about video games: titles, platforms, release dates, publishers and genres.

Follow this workflow for every question about games:
1. Call retrieve_game to search the local database.
2. Call evaluate_retrieval with the user's question and the retrieved documents.
   Pass every retrieved game exactly as returned (all fields, as a JSON string), without
   summarising or removing anything.
3. If the evaluation says the documents are useful, answer from them.
4. If they are not useful, call game_web_search.
5. If the web search found a clear, specific fact about a game that is NOT already in the
   local results, call store_game_fact once for that game so it is available locally next time.
   Call it at most once per game. Use the platform the game was officially released on.
   Never store guesses. If it is refused because the fact check failed, you may call it
   one more time with the corrected values given in the reason; otherwise do not retry.

Rules:
- Base your answer only on tool results, never on your own memory.
- Be concise and state the answer directly first, then the supporting detail.
- Say whether the answer came from the local database or the web, and cite your sources:
  the "citation" of each local record you used, and the URLs you used from the web.
- If you cannot find a reliable answer, say so instead of guessing.
- For follow-up questions, use the conversation history to understand what the user refers to.
"""

udaplay_agent = Agent(
    model_name=MODEL_NAME,
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search, store_game_fact],
    temperature=TEMPERATURE,
)

#### Structured output

The agent's final reply is text. `ask_udaplay` turns it into a `GameAnswer` object.

- **`print_trace`** shows each tool call and a shortened result, so you can see *why* the agent chose
  its path (for example, what the evaluator said before a web search).
- **`source`** is `local_db`, `web`, or `conversation` when the agent answered a follow-up from the
  session history without calling any tools.
- **`tools_used` is computed in Python** from the messages of this run, not guessed by the LLM, so it is
  always accurate. `source` is derived from it the same way.
- **Citations are checked**: only URLs that actually appear in the tool outputs are kept, so the
  model can't invent a source.
- **`answer`, `confidence` and `citations`** are extracted by a second LLM call with `GameAnswer` as the
  `response_format`, which guarantees valid, typed output.
- Only the messages from the **current** question are used. The agent keeps the whole conversation in
  memory, so older tool results would otherwise leak into this answer's sources.
- `ask_udaplay` accepts an optional `agent` and `agent_input`, so the long-term memory agent in the
  Advanced section can reuse the same structured output without duplicating this code.
- **`reasoning`** explains in 1-3 sentences how the answer was reached (what was found locally, what the
  evaluator decided, whether and why the web was searched). The trace above each answer shows the
  full evidence: every tool call and result, with the evaluator's explanation printed in full.
- **`citations`** cover both kinds of source: the `citation` of each local record used and the URLs of
  web results used. Both are checked against the tool outputs, so nothing can be invented, and a
  citation must be a URL or a `Local DB:` / `Learned from web:` reference (not a copied sentence).
  If a web answer ends up with no URL, the web search results' URLs are cited directly, so a web
  answer is never uncited.
- The formatter is told **which tools ran** (or "none"), so the reasoning for a follow-up answered from
  the conversation doesn't claim it came from the database.

In [14]:
class GameAnswer(BaseModel):
    question: str
    answer: str = Field(description="Direct, concise answer to the question")
    source: Literal["local_db", "web", "conversation", "memory"] = Field(description="Where the answer came from")
    confidence: Literal["high", "medium", "low"] = Field(description="How well the sources support the answer")
    reasoning: str = Field(description=(
        "1-3 sentences on how the answer was reached: what the local search found, what the evaluator "
        "concluded, and whether and why the web was searched"))
    citations: list[str] = Field(description=(
        "Sources used: the exact 'citation' text of each local record used, and URLs from web results used"))
    tools_used: list[str] = Field(description="Tools called while answering, in order")


def is_valid_citation(citation: str, sources: str) -> bool:
    """A citation must be a URL or a local record reference, and must appear in the sources."""
    looks_right = citation.startswith(("http://", "https://", "Local DB:", "Learned from web:"))
    return bool(citation) and looks_right and citation in sources


def _messages_for_this_turn(messages: list) -> list:
    """Return only the messages after the latest user question."""
    last_user_idx = max(i for i, m in enumerate(messages) if isinstance(m, UserMessage))
    return messages[last_user_idx:]


def _tool_text(m: ToolMessage) -> str:
    """Tool results are stored JSON-encoded; decode them so accents etc. read normally."""
    try:
        return json.loads(m.content)
    except (TypeError, ValueError):
        return m.content


def print_trace(turn_messages: list, max_len: int = 300):
    """
    Print the agent's reasoning trail: its own notes, each tool call, and each result.
    Results are shortened, except the evaluator's, which is the reasoning behind the next step.
    """
    for m in turn_messages:
        if isinstance(m, AIMessage) and m.tool_calls:
            if m.content:
                print(f"  [agent] {m.content}")
            for call in m.tool_calls:
                print(f"  -> {call.function.name}({call.function.arguments[:max_len]})")
        elif isinstance(m, ToolMessage):
            text = _tool_text(m)
            print(f"  <- {m.name}: {text if m.name == 'evaluate_retrieval' else text[:max_len]}")


def ask_udaplay(question: str, session_id: str = "default", verbose: bool = True,
                agent: Agent = None, agent_input: str = None) -> GameAnswer:
    """
    Run the agent on a question and return a structured GameAnswer.
    - agent: which agent to use (defaults to udaplay_agent)
    - agent_input: text actually sent to the agent, if different from the question
      (used by the memory agent to add what it knows about the user)
    """
    agent = agent or udaplay_agent
    run = agent.invoke(agent_input or question, session_id=session_id)
    turn_messages = _messages_for_this_turn(run.get_final_state()["messages"])
    if verbose:
        print_trace(turn_messages)

    # Tools called in this turn, in order (computed, not guessed)
    tools_used = [
        call.function.name
        for m in turn_messages
        if isinstance(m, AIMessage) and m.tool_calls
        for call in m.tool_calls
    ]
    if "game_web_search" in tools_used:
        source = "web"
    elif "retrieve_game" in tools_used:
        source = "local_db"
    else:
        source = "conversation"  # answered from earlier messages in the session

    final_text = turn_messages[-1].content or ""
    tool_outputs = "\n".join(_tool_text(m) for m in turn_messages if isinstance(m, ToolMessage))

    formatter = LLM(model=MODEL_NAME, temperature=0.0)
    response = formatter.invoke(
        [
            SystemMessage(content=(
                "Convert the assistant's answer into the requested structure. "
                "Use only information in the answer and tool outputs. "
                "In reasoning, explain in 1-3 sentences how the answer was reached, based on the tool "
                "outputs (what the local search found, what the evaluator concluded, whether the web was "
                "searched and why). If there are no tool outputs, say it was answered from the conversation. "
                "In citations, list the exact 'citation' text of each local record the answer relies on, "
                "and the URLs of web results it relies on. Only use citations and URLs that appear in the "
                "tool outputs. "
                "Set confidence to high if the sources directly state the answer, "
                "medium if it needed some interpretation, low if the sources were weak or missing."
            )),
            UserMessage(content=(
                f"Question: {question}\n\n"
                f"Tools called in this turn: {', '.join(tools_used) if tools_used else 'none (answered from the conversation history)'}\n\n"
                f"Assistant answer: {final_text}\n\n"
                f"Tool outputs: {tool_outputs}"
            )),
        ],
        response_format=GameAnswer,
    )
    result = GameAnswer.model_validate_json(response.content)
    # Overwrite with the values we know for certain
    result.question = question
    result.tools_used = tools_used
    result.source = source
    # Keep only citations and URLs that really appear in the tool outputs (nothing invented), without duplicates
    result.citations = [c for c in dict.fromkeys(result.citations) if is_valid_citation(c, tool_outputs)]
    # A web answer must cite its sources: if no URL survived, cite the web search results directly
    if source == "web" and not any(c.startswith("http") for c in result.citations):
        web_urls = [r["url"] for r in LAST_WEB_RESULTS.get("results", []) if r.get("url")]
        result.citations += list(dict.fromkeys(web_urls))[:3]
    return result


def show(result: GameAnswer):
    """Print a GameAnswer in a readable way."""
    print(json.dumps(result.model_dump(), indent=2, ensure_ascii=False))

### Invoke the agent

The three questions cover the three paths through the agent:

1. **Pokémon Gold and Silver**: in the local DB, answered without the web.
2. **First 3D Mario platformer**: in the local DB, but needs reasoning over the results.
3. **Mortal Kombat X on PS5**: not in the local DB, so it should search the web and store the fact.

Each question uses its own `session_id`, so the questions don't influence each other.

In [15]:
show(ask_udaplay("When was Pokémon Gold and Silver released?", session_id="q1"))

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
  -> retrieve_game({"query":"Pokémon Gold and Silver release date"})
  <- retrieve_game: [{'YearOfRelease': 1999, 'Genre': 'Role-playing', 'Name': 'Pokémon Gold and Silver', 'Publisher': 'Nintendo', 'Platform': 'Game Boy Color', 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', 'distance': 0.255, 'citation': 'Local DB: Pokémon Go
  -> evaluate_retrieval({"question":"When was Pokémon Gold and Silver released?","retrieved_docs":["{'YearOfRelease': 1999, 'Genre': 'Role-playing', 'Name': 'Pokémon Gold and Silver', 'Publisher': 'Nintendo', 'Platform': 'Game Boy Color', 'Description': 'Second-

In [16]:
show(ask_udaplay("Which one was the first 3D platformer Mario game?", session_id="q2"))

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
  -> retrieve_game({"query":"first 3D platformer Mario game"})
  <- retrieve_game: [{'Platform': 'Nintendo 64', 'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", 'Genre': 'Platformer', 'Name': 'Super Mario 64', 'YearOfRelease': 1996, 'Publisher': 'Nintendo', 'distance': 0.206, 'citation': 'Local
  -> evaluate_retrieval({"question":"Which one was the first 3D platformer Mario game?","retrieved_docs":["{\"Platform\":\"Nintendo 64\",\"Description\":\"A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess P

In [17]:
show(ask_udaplay("Was Mortal Kombat X released for PlayStation 5?", session_id="q3"))

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
  -> retrieve_game({"query":"Mortal Kombat X PlayStation 5 release"})
  <- retrieve_game: [{'Publisher': 'Sony Interactive Entertainment', 'Platform': 'PlayStation 5', 'Genre': 'Action-adventure', 'Name': "Marvel's Spider-Man 2", 'Description': 'The sequel to the acclaimed Spider-Man game, featuring both Peter Parker and Miles Morales as playable characters.', 'YearOfRelease': 2023, 'dis
  -> evaluate_retrieval({"question":"Was Mortal Kombat X released for PlayStation

#### Conversation state

A follow-up in the **same session** (`q2`) only makes sense if the agent remembers the previous question.

In [18]:
show(ask_udaplay("What platform was it released on, and who published it?", session_id="q2"))

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
{
  "question": "What platform was it released on, and who published it?",
  "answer": "Super Mario 64 was released on the Nintendo 64 and published by Nintendo.",
  "source": "conversation",
  "confidence": "high",
  "reasoning": "The answer was derived from the local database, which provided specific details about the release platform and publisher of Super Mario 64.",
  "citations": [],
  "tools_used": []
}


#### Stored knowledge

After the Mortal Kombat X question, the fact found on the web should now be in the local DB.
Asking again in a new session should find it locally. Whether the judge accepts it as enough to
answer (and skips the web) depends on what was stored, which the trace shows.

In [19]:
# Facts saved from the web
print(web_collection.get(include=["documents"])["documents"])

show(ask_udaplay("Was Mortal Kombat X released for PlayStation 5?", session_id="q4"))

['[PlayStation 4] Mortal Kombat X (2015) - Mortal Kombat X was originally released for PlayStation 4 on April 14, 2015, and is playable on PlayStation 5 through backward compatibility.']
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
  -> retrieve_game({"query":"Mortal Kombat X PlayStation 5 release"})
  <- retrieve_game: [{'SourceURL': 'https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V', 'Publisher': 'Warner Bros. Interactive Entertainment', 'Description': 'Mortal Kombat X was originally released for PlayStation 4 on April 14, 2015, and is playable on Pl
  -> evaluate_retrieval({"question":"Was Mortal Kombat X released for Play

### (Optional) Advanced

### Long-term memory

The agent already remembers two things: the **conversation** (per `session_id`, lost when the session
ends) and **facts about games** learned from the web (`udaplay_web`). Long-term memory adds a third:
**facts about the user**, such as the consoles they own or the genres they like. These persist across
sessions and restarts, so the agent can personalise answers in a brand-new conversation.

**Design choices**

- **Stored in its own persistent collection (`udaplay_memory`).** `LongTermMemory` in `lib/memory.py`
  keeps memories in an in-memory database and deletes them every time it is created, so nothing would
  survive a restart. We reuse its `MemoryFragment` format (content, owner, namespace, timestamp) but
  store the fragments on disk with the same embedding model as the rest of the project.
- **Every memory has an owner (`user_id`), separate from `session_id`.** A user can have many sessions;
  memories belong to the user, not the conversation. Searches are filtered by owner, so one user never
  sees another user's memories.
- **The user ID is set in code, never chosen by the model.** Tools only receive what the model passes
  them, and a model-supplied user ID could be wrong or point at someone else's memories. So
  `ask_with_memory` sets `CURRENT_USER` and the tool reads it.
- **Saving is a tool; recalling is automatic.** Deciding *what* is worth remembering needs judgement
  (a lasting preference vs. a one-off question), so the model decides when to call
  `remember_user_preference`. Recalling should happen *every* time, so code does it: before each
  question, the user's most relevant memories are looked up and added to the message. The agent can't
  forget to check, and it costs no extra LLM call.
- **Stable IDs from owner + text**, with `upsert`, so saving the same preference twice doesn't create
  a duplicate.

In [20]:
from lib.memory import MemoryFragment

# Persistent collection for facts about users
memory_collection = chroma_client.get_or_create_collection(
    name="udaplay_memory",
    embedding_function=embedding_fn,
)

# Set by ask_with_memory before each run; read by the memory tool (never chosen by the model)
CURRENT_USER = None


def recall_memories(user_id: str, query: str, limit: int = 5) -> list[str]:
    """Return this user's memories most relevant to the query (newest format: plain text)."""
    owned = memory_collection.get(where={"owner": user_id})
    if not owned["ids"]:
        return []
    results = memory_collection.query(
        query_texts=[query],
        n_results=min(limit, len(owned["ids"])),  # never ask for more than the user has
        where={"owner": user_id},
    )
    return results["documents"][0]


@tool
def remember_user_preference(fact: str) -> dict:
    """
    Saves a lasting fact about the current user to long-term memory, so it can be used
    in future conversations.
    Use it for preferences and personal context, e.g. consoles they own, favourite genres,
    games they have played or disliked. Do not use it for questions or one-off requests.
    args:
    - fact: a short statement about the user in the third person, e.g. "Owns a Nintendo Switch"
    """
    if not CURRENT_USER:
        return {"stored": False, "error": "No user is set for this conversation."}
    try:
        fragment = MemoryFragment(content=fact.strip(), owner=CURRENT_USER, namespace="preferences")
        key = f"{fragment.owner}|{fragment.content.lower()}".encode("utf-8")
        memory_id = "mem-" + hashlib.md5(key).hexdigest()[:12]
        memory_collection.upsert(
            ids=[memory_id],
            documents=[fragment.content],
            metadatas=[{
                "owner": fragment.owner,
                "namespace": fragment.namespace,
                "timestamp": fragment.timestamp,
            }],
        )
        return {"stored": True, "id": memory_id}
    except Exception as e:
        return {"stored": False, "error": f"Could not save memory: {e}"}

#### Memory-aware agent

A second agent with the same tools plus `remember_user_preference`, and extra instructions on how to
use memory. The original `udaplay_agent` above is left unchanged.

`ask_with_memory` sets the current user, recalls their memories, adds them to the message, and reuses
`ask_udaplay` for the structured output. The result also lists `memories_used`, and `source` is
`memory` when the answer came from memory alone.

If nothing is known about the user's consoles, the agent is told to **ask** rather than guess. In an
earlier run, a new user was recommended a PlayStation 1 game without being asked what they play.

In [21]:
MEMORY_INSTRUCTIONS = """
Long-term memory:
- Messages may end with a section "Known about this user", recalled from earlier conversations.
  Use it to personalise answers (e.g. only recommend games for consoles they own) and say when you did.
- When the user shares a lasting fact about themselves (consoles owned, favourite genres,
  games played or disliked), call remember_user_preference once per fact, written as a short
  third-person statement such as "Owns a Nintendo Switch".
- Do not save questions, one-off requests, or anything sensitive.
- If the user only shares information about themselves, save it and briefly acknowledge it;
  there is no need to search for games.
- For recommendations, only recommend games returned by retrieve_game or game_web_search.
- If the user asks for a game they can play but nothing is known about which consoles they own,
  ask them which console(s) they have instead of guessing. Do not call any tools in that case.
"""

memory_agent = Agent(
    model_name=MODEL_NAME,
    instructions=INSTRUCTIONS + MEMORY_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search, store_game_fact, remember_user_preference],
    temperature=TEMPERATURE,
)


class MemoryAnswer(GameAnswer):
    user_id: str
    memories_used: list[str]


def ask_with_memory(question: str, user_id: str, session_id: str, verbose: bool = True) -> MemoryAnswer:
    """Ask the memory-aware agent a question on behalf of a specific user."""
    global CURRENT_USER
    CURRENT_USER = user_id

    memories = recall_memories(user_id, question)
    agent_input = question
    if memories:
        known = "\n".join(f"- {m}" for m in memories)
        agent_input = f"{question}\n\nKnown about this user:\n{known}"
    if verbose:
        print(f"  [memory] recalled for {user_id}: {memories}")

    result = ask_udaplay(question, session_id=session_id, verbose=verbose,
                         agent=memory_agent, agent_input=agent_input)

    # Answered without any game tools but with recalled memories -> it came from memory
    if result.source == "conversation" and memories:
        result.source = "memory"
    return MemoryAnswer(**result.model_dump(), user_id=user_id, memories_used=memories)

#### Memory demo

1. **Reset** the memory collection so the demo is reproducible.
2. **Session `m1`**: William tells the agent about himself. It should call `remember_user_preference`.
3. **Session `m2`**: a *new* session for William, so there is no chat history. The recommendation
   should still use what was remembered (Switch owner → Mario Kart 8 Deluxe).
4. **Session `m3`**: the same question from a different user. Nothing is recalled, so instead of
   guessing, the agent should ask which console they own. This shows memories are kept per user.

In [22]:
chroma_client.delete_collection(name="udaplay_memory")
memory_collection = chroma_client.create_collection(name="udaplay_memory", embedding_function=embedding_fn)
print(f"Memories: {memory_collection.count()}")  # expected: 0

Memories: 0


In [23]:
show(ask_with_memory(
    "I'm a big Nintendo fan and the only console I own is a Nintendo Switch. I love racing games.",
    user_id="william", session_id="m1",
))

  [memory] recalled for william: []
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
  -> remember_user_preference({"fact": "Owns a Nintendo Switch"})
  -> remember_user_preference({"fact": "Loves racing games"})
  <- remember_user_preference: {'stored': True, 'id': 'mem-0f0676419a65'}
  <- remember_user_preference: {'stored': True, 'id': 'mem-4379c303e752'}
{
  "question": "I'm a big Nintendo fan and the only console I own is a Nintendo Switch. I love racing games.",
  "answer": "Got it! You're a big Nintendo fan, own a Nintendo Switch, and love racing games.",
  "source": "conversation",
  "confidence": "high",
  "reasoning": "The answer was derived from the conversation where the user's preferences were noted and stored. No additional sources were needed as the information was 

In [24]:
# What was saved for William
print(memory_collection.get(where={"owner": "william"}, include=["documents"])["documents"])

['Owns a Nintendo Switch', 'Loves racing games']


In [25]:
show(ask_with_memory(
    "Can you recommend a game from your catalog that I can play?",
    user_id="william", session_id="m2",
))

  [memory] recalled for william: ['Loves racing games', 'Owns a Nintendo Switch']
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
  -> retrieve_game({"query":"racing games for Nintendo Switch"})
  <- retrieve_game: [{'Publisher': 'Nintendo', 'Name': 'Mario Kart 8 Deluxe', 'YearOfRelease': 2017, 'Genre': 'Racing', 'Description': 'An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.', 'Platform': 'Nintendo Switch', 'distance': 0.278, 'citation': 'Local DB: Mario
  -> evaluate_retrieval({"question":"Can you recommend a game from your catalog that I can play?","retrieved_docs":["{\"Publisher\":\"Nintendo\",\"Name\":\"Mario Kart 8 Deluxe\",\"Year

In [26]:
show(ask_with_memory(
    "Can you recommend a game from your catalog that I can play?",
    user_id="guest", session_id="m3",
))

  [memory] recalled for guest: []
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
{
  "question": "Can you recommend a game from your catalog that I can play?",
  "answer": "Could you please let me know which console(s) you own? This will help me recommend a game that you can play.",
  "source": "conversation",
  "confidence": "high",
  "reasoning": "The answer was reached based on the conversation, as the assistant asked for more information about the user's console ownership to provide a relevant game recommendation.",
  "citations": [],
  "tools_used": [],
  "user_id": "guest",
  "memories_used": []
}


### State machine: the tools as fixed pipeline steps

The agent above decides *which tool to call next* on every turn. Here the same tools become
**pre-defined steps** of a `StateMachine` (from `lib/state_machine.py`, the same engine the `Agent`
class runs on), connected in a fixed order:

```
entry → retrieve → evaluate ─┬─ useful ─────────────────────────→ answer → end
                             └─ not useful → web_search → store ─→ answer → end
```

**Design choices**

- **The tool functions are reused as-is.** Each step calls the existing `retrieve_game`,
  `evaluate_retrieval`, `game_web_search` and `store_game_fact`, so both versions share the same
  logic, error handling, duplicate guard and fact check. Only the *control flow* differs.
- **The branch is plain Python**, not an LLM decision: the `evaluate` step's `useful` flag picks the
  next step. The evaluation step can never be skipped, and choosing the next step costs no tokens.
- **Code passes data between steps.** The agent had to copy the retrieved documents into the
  evaluator call itself (which may be why it was once given the wrong documents); here the `evaluate`
  step receives exactly what `retrieve` returned.
- **The LLM is only used where judgement is needed:** the evaluator, extracting a fact from the web
  results to store, and writing the final answer. The answer step uses a structured `response_format`,
  so the structured output comes straight from that one call instead of a separate formatting call.
- **The verified fact is passed to the answer step.** In the first run the pipeline answered
  "Yes, Mortal Kombat X is playable on PlayStation 5": the store step had extracted and fact-checked
  the right fact (released on PS4 in 2015), but the answer step only saw the raw web results, which
  talk about PS5 availability. The agent didn't make this mistake because the fact check was part of
  its conversation. Now the answer step receives the verified fact (or the fact check's correction)
  and is told explicitly that being playable is not the same as being released. The first answers
  were also bare ("1999"), so the prompt now asks for full sentences with the key detail.
- **The state records the path taken** (`path`), and every step is saved as a snapshot in the `Run`,
  so each run can be inspected afterwards.
- **Trade-off:** the pipeline has no conversation history and always runs the same steps, so it
  can't handle follow-up questions ("what platform was *it* on?") or small talk. The comparison below
  measures what that rigidity buys.

In [27]:
from typing import TypedDict
from lib.state_machine import StateMachine, Step, EntryPoint, Termination


class PipelineState(TypedDict):
    question: str          # the user's question
    retrieved_docs: list   # results of retrieve_game
    evaluation: dict       # result of evaluate_retrieval
    web_results: dict      # result of game_web_search (empty if not needed)
    stored_fact: dict      # result of store_game_fact (empty if nothing was stored)
    answer: dict           # final structured answer
    path: list             # names of the steps taken, in order


class GameFact(BaseModel):
    found: bool = Field(description="Whether the web results clearly state a release fact about the game asked about")
    name: str
    platform: str = Field(description="Platform the game was officially released on, not one it is only playable on")
    year_of_release: int
    description: str = Field(description="Short description including the key fact that was found")
    genre: str = Field(description='Genre, or "Unknown"')
    publisher: str = Field(description='Publisher, or "Unknown"')
    source_url: str = Field(description="URL of the result that supports the fact")


class PipelineAnswer(BaseModel):
    answer: str = Field(description="Direct, concise answer to the question")
    confidence: Literal["high", "medium", "low"] = Field(description="How well the context supports the answer")
    reasoning: str = Field(description="1-2 sentences on how the answer follows from the context")
    citations: list[str] = Field(description=(
        "Sources used: the exact 'citation' text of each local record used, and URLs from web results used"))

#### Steps

Each step is a small function that takes the state and returns only the fields it changes.

In [28]:
def retrieve_step(state: PipelineState) -> dict:
    """Search the local vector DB (original games + learned facts)."""
    docs = retrieve_game(state["question"])
    return {"retrieved_docs": docs, "path": state["path"] + ["retrieve"]}


def evaluate_step(state: PipelineState) -> dict:
    """Judge whether the retrieved documents answer the question."""
    docs = [json.dumps(d, ensure_ascii=False) for d in state["retrieved_docs"]]
    evaluation = evaluate_retrieval(state["question"], docs)
    return {"evaluation": evaluation, "path": state["path"] + ["evaluate"]}


def web_search_step(state: PipelineState) -> dict:
    """Search the web when local knowledge is not enough."""
    results = game_web_search(state["question"])
    return {"web_results": results, "path": state["path"] + ["web_search"]}


def store_step(state: PipelineState) -> dict:
    """Extract one release fact from the web results and store it (fact-checked by store_game_fact)."""
    stored = {"stored": False, "reason": "No clear fact found in the web results."}
    try:
        extractor = LLM(model=MODEL_NAME, temperature=0.0)
        response = extractor.invoke(
            [
                SystemMessage(content=(
                    "Extract the main release fact about the game the user asked about from the web results. "
                    "Use the platform the game was officially released on, not a platform where it is only "
                    "playable through backward compatibility or a subscription collection. "
                    "Set found to false if the results don't clearly state it."
                )),
                UserMessage(content=(
                    f"Question: {state['question']}\n\n"
                    f"Web results: {json.dumps(state['web_results'], ensure_ascii=False)}"
                )),
            ],
            response_format=GameFact,
        )
        fact = GameFact.model_validate_json(response.content)
        if fact.found:
            stored = store_game_fact(**fact.model_dump(exclude={"found"}))
            # Keep the fact itself, so the answer step can use it
            stored["fact"] = f"{fact.name} was released on {fact.platform} in {fact.year_of_release}."
    except Exception as e:
        stored = {"stored": False, "error": f"Could not extract fact: {e}"}
    return {"stored_fact": stored, "path": state["path"] + ["store"]}


def answer_step(state: PipelineState) -> dict:
    """Write the final structured answer from whichever context the pipeline gathered."""
    if state.get("web_results"):
        context = json.dumps(state["web_results"], ensure_ascii=False)
        stored = state.get("stored_fact", {})
        if stored.get("stored"):
            # The fact passed the fact check in store_game_fact
            context = f"Verified fact: {stored['fact']}\n\nWeb results: {context}"
        elif stored.get("reason", "").startswith("Fact check failed"):
            # The fact check's reason contains the corrected values
            context = f"Fact check note: {stored['reason']}\n\nWeb results: {context}"
    else:
        context = json.dumps(state["retrieved_docs"], ensure_ascii=False)

    writer = LLM(model=MODEL_NAME, temperature=0.0)
    response = writer.invoke(
        [
            SystemMessage(content=(
                "You are UdaPlay, a video game research assistant. Answer the exact question asked, "
                "using only the context. Answer in one or two full sentences: the direct answer first, "
                "then the key supporting detail (such as the platform and year). "
                "A game that is only playable on a platform through backward compatibility, streaming or "
                "a subscription collection was NOT released for that platform; say so explicitly. "
                "If a verified fact is given, rely on it. "
                "Cite the exact 'citation' text of each local record you used, and the URLs of web results you used. "
                "If the context doesn't answer the question, say so instead of guessing."
            )),
            UserMessage(content=f"Question: {state['question']}\n\nContext: {context}"),
        ],
        response_format=PipelineAnswer,
    )
    answer = PipelineAnswer.model_validate_json(response.content)
    # Keep only citations and URLs that really appear in the context (no invented sources)
    answer.citations = [c for c in dict.fromkeys(answer.citations) if is_valid_citation(c, context)]
    return {"answer": answer.model_dump(), "path": state["path"] + ["answer"]}

#### Wiring the state machine

`evaluate` has two possible next steps. The condition function reads the `useful` flag and returns
the step to run next.

In [29]:
def build_pipeline() -> StateMachine[PipelineState]:
    machine = StateMachine[PipelineState](PipelineState)

    entry = EntryPoint[PipelineState]()
    retrieve = Step[PipelineState]("retrieve", retrieve_step)
    evaluate = Step[PipelineState]("evaluate", evaluate_step)
    web_search = Step[PipelineState]("web_search", web_search_step)
    store = Step[PipelineState]("store", store_step)
    answer = Step[PipelineState]("answer", answer_step)
    termination = Termination[PipelineState]()

    machine.add_steps([entry, retrieve, evaluate, web_search, store, answer, termination])

    machine.connect(entry, retrieve)
    machine.connect(retrieve, evaluate)

    def route_after_evaluation(state: PipelineState):
        """Answer from local knowledge if it was judged useful, otherwise search the web."""
        return answer if state["evaluation"].get("useful") else web_search

    machine.connect(evaluate, [answer, web_search], route_after_evaluation)
    machine.connect(web_search, store)
    machine.connect(store, answer)
    machine.connect(answer, termination)
    return machine


udaplay_pipeline = build_pipeline()


def ask_pipeline(question: str, verbose: bool = True) -> dict:
    """Run the state-machine pipeline and return a structured result like GameAnswer."""
    run = udaplay_pipeline.run({
        "question": question,
        "retrieved_docs": [],
        "evaluation": {},
        "web_results": {},
        "stored_fact": {},
        "answer": {},
        "path": [],
    })
    state = run.get_final_state()
    result = {
        "question": question,
        **state["answer"],
        "source": "web" if "web_search" in state["path"] else "local_db",
        "path": state["path"],
    }
    if verbose:
        print(f"  path: {' > '.join(state['path'])}")
        print(f"  evaluation: useful={state['evaluation'].get('useful')} - {state['evaluation'].get('description', '')}")
        if state["stored_fact"]:
            print(f"  stored: {state['stored_fact']}")
    return result

#### Run the pipeline

The same three questions as the agent. The learned facts are reset first, so the Mortal Kombat X
question has to go through the web branch again (and the store step, with its fact check).

In [30]:
def reset_learned_facts():
    """Drop and recreate udaplay_web, so a run starts from the 15 original games."""
    global web_collection
    chroma_client.delete_collection(name="udaplay_web")
    web_collection = chroma_client.create_collection(name="udaplay_web", embedding_function=embedding_fn)


reset_learned_facts()
for q in [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]:
    print(f"\nQ: {q}")
    print(json.dumps(ask_pipeline(q), indent=2, ensure_ascii=False))


Q: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__
  path: retrieve > evaluate > answer
  evaluation: useful=True - The documents contain the necessary information to answer the question about the release date of Pokémon Gold and Silver. Specifically, one of the documents explicitly states that Pokémon Gold and Silver was released in 1999, which is sufficient to respond to the query. There are no other games or platforms mentioned that would confuse the answer, and the relevant details are clearly presented.
{
  "question": "When was Pokémon Gold and Silver released?",
  "answer": "Pokémon Gold and Silver was released in 1999.",
  "confidence": "high",
  "reasoning": "The context explicitly states the release year for Pokémon Gold and Silver as 1999, along with its platform, Game Boy Color.",
  "

#### Comparison: agent vs. state machine

Both versions answer the same questions, each starting from the same knowledge (learned facts reset
before each system). For every run we measure:

- **LLM calls and tokens**: counted by wrapping `LLM.invoke`, so *every* call is included (the agent's
  own turns, the evaluator, the fact check, the answer formatter). The wrapper only counts; it doesn't
  change behaviour, and it is applied once even if this cell is re-run.
- **Time** in seconds, and the **path** (tools or steps) each system took.

In [31]:
import time

USAGE = {"llm_calls": 0, "tokens": 0}

if not getattr(LLM.invoke, "_counts_usage", False):
    _original_invoke = LLM.invoke

    def _counting_invoke(self, *args, **kwargs):
        response = _original_invoke(self, *args, **kwargs)
        USAGE["llm_calls"] += 1
        if response.token_usage:
            USAGE["tokens"] += response.token_usage.total_tokens
        return response

    _counting_invoke._counts_usage = True
    LLM.invoke = _counting_invoke


def measure(system: str, question: str, fn) -> dict:
    """Run one question through one system and record cost, time and path."""
    USAGE.update(llm_calls=0, tokens=0)
    start = time.time()
    result = fn(question)
    return {
        "system": system,
        "question": question,
        "answer": result["answer"],
        "source": result["source"],
        "path": result.get("tools_used") or result.get("path"),
        "llm_calls": USAGE["llm_calls"],
        "tokens": USAGE["tokens"],
        "seconds": round(time.time() - start, 1),
    }


QUESTIONS = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

rows = []
reset_learned_facts()
for i, q in enumerate(QUESTIONS):
    rows.append(measure("agent", q, lambda q: ask_udaplay(q, session_id=f"cmp-{i}", verbose=False).model_dump()))
reset_learned_facts()
for q in QUESTIONS:
    rows.append(measure("state machine", q, lambda q: ask_pipeline(q, verbose=False)))

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor


In [32]:
# Results table
header = f"{'system':<14} {'question':<32} {'source':<9} {'calls':>5} {'tokens':>7} {'secs':>5}  path"
print(header)
print("-" * len(header))
for r in rows:
    print(f"{r['system']:<14} {r['question'][:31]:<32} {r['source']:<9} {r['llm_calls']:>5} "
          f"{r['tokens']:>7} {r['seconds']:>5}  {' > '.join(r['path'])}")

print()
for system in ["agent", "state machine"]:
    sel = [r for r in rows if r["system"] == system]
    print(f"{system:<14} total: {sum(r['llm_calls'] for r in sel)} LLM calls, "
          f"{sum(r['tokens'] for r in sel)} tokens, {round(sum(r['seconds'] for r in sel), 1)} s")

print("\nAnswers:")
for r in rows:
    print(f"- [{r['system']}] {r['question']}\n    {r['answer']}")

system         question                         source    calls  tokens  secs  path
-----------------------------------------------------------------------------------
agent          When was Pokémon Gold and Silve  local_db      5    5023  10.0  retrieve_game > evaluate_retrieval
agent          Which one was the first 3D plat  local_db      5    5116   9.2  retrieve_game > evaluate_retrieval
agent          Was Mortal Kombat X released fo  web           8   13413  21.3  retrieve_game > evaluate_retrieval > game_web_search > store_game_fact
state machine  When was Pokémon Gold and Silve  local_db      2    1213   3.8  retrieve > evaluate > answer
state machine  Which one was the first 3D plat  local_db      2    1305   4.7  retrieve > evaluate > answer
state machine  Was Mortal Kombat X released fo  web           4    4385   9.2  retrieve > evaluate > web_search > store > answer

agent          total: 18 LLM calls, 23552 tokens, 40.5 s
state machine  total: 8 LLM calls, 6903 tokens, 17.

#### Findings

Measured in the final run (numbers vary slightly from run to run):

| | Agent | State machine |
|---|---|---|
| LLM calls (3 questions) | 18 | 8 |
| Tokens | 23,987 | 6,862 |
| Time | 36.8 s | 16.9 s |

**Cost and speed.** The state machine used about **56% fewer LLM calls, 71% fewer tokens and less than
half the time**. For a question answered locally, the agent needs 5 calls (three of its own turns to
pick tools and write the answer, plus the evaluator and the output formatter), while the pipeline needs
2 (evaluator and answer), because the next step is chosen in code. The agent's calls are also larger:
every turn re-sends the whole conversation, including the growing tool results.

**Accuracy.** Both got Pokémon Gold and Silver (1999) and Super Mario 64 right. On Mortal Kombat X the
first version of the pipeline answered "Yes, it is playable on PS5", which dodges the question. The
research was right, but the verified fact never reached the answer step. The agent kept everything in
one conversation, so it could use its own fact check when writing the answer. This shows the main risk
of a fixed pipeline: **each step only knows what the state passes to it**, so the state has to be
designed deliberately. After passing the verified fact to the answer step, the pipeline answered
correctly: "Mortal Kombat X was not released for PlayStation 5; it was originally released for
PlayStation 4 in 2015." Both systems now give correct answers to all three questions; the agent's
answers are a little richer (e.g. it adds the platform and year for Super Mario 64).

**Where each design fits.**

- **State machine:** predictable, known-shape questions. It is cheaper, faster, easy to debug (the
  path and every snapshot are recorded), and it can never skip the evaluation step.
- **Agent:** open-ended conversations. It handles follow-ups ("what platform was *it* on?"), users
  sharing information about themselves, and questions that don't fit a fixed path. That flexibility
  costs about 3.5× the tokens and relies on the model following its instructions.

A practical combination would be to route known question types through the pipeline and fall back to
the agent for everything else.